# Exercice 1.5: Feature Engineering : transformation et encodage sur Ames Housing

**Séance 5: Feature Engineering : transformation et encodage**

---



## Contexte

Jusqu'ici vous avez exploré, visualisé et compris les données. Maintenant il faut les **transformer** pour les rendre exploitables par un algorithme de machine learning.

Un modèle ML ne comprend que des nombres. La variable `Neighborhood` (quartier), `ExterQual` (qualité extérieure), les valeurs manquantes dans `BsmtQual`: tout cela doit être encodé, imputed, mis à l'échelle. Et chaque choix a un impact mesurable sur la performance.

Dans cet exercice, vous allez faire le lien entre votre EDA et le feature engineering : les distributions asymétriques que vous avez observées, les variables catégorielles à haute cardinalité, les valeurs manquantes structurelles: chacune appelle une stratégie spécifique. À la fin, vous comparerez quantitativement un pipeline brut et un pipeline avec feature engineering complet sur un modèle de machine learning.

---



## Objectifs d'apprentissage

- Faire le lien entre les observations de l'EDA et les transformations à appliquer
- Choisir et appliquer la méthode d'encodage adaptée à chaque type de variable catégorielle
- Transformer les distributions asymétriques pour améliorer la modélisation
- Gérer les valeurs manquantes sans fuir de données
- Mettre à l'échelle les variables et comprendre le comportement de chaque scaler face aux outliers
- Évaluer l'impact des transformations via un modèle Lasso baseline

---



## Règle fondamentale : pas de fuite de données

> Toutes les transformations (imputation, encoding, scaling) doivent être **apprises uniquement sur le train set** et **appliquées** ensuite sur le test set. Apprendre une transformation sur l'ensemble du dataset, c'est laisser le modèle "voir" les données de test pendant l'entraînement: c'est de la fuite de données.
>
> La solution : utiliser `fit()` sur le train, `transform()` sur le test. Ou mieux : encapsuler tout dans un `Pipeline` sklearn.

---



## Section 1: Imports, chargement et split

**Ce que vous devez faire :**
- Importez les bibliothèques nécessaires. Cette séance introduit plusieurs modules scikit-learn que vous n'avez (peut-être) pas encore utilisés : `OneHotEncoder`, `OrdinalEncoder`, `TargetEncoder`, `StandardScaler`, `RobustScaler`, `MinMaxScaler`, `SimpleImputer`, `KNNImputer`, `ColumnTransformer`, `Pipeline`, `Lasso`.
- Chargez Ames Housing **depuis le fichier brut**: pas depuis une version nettoyée des exercices précédents. Le feature engineering doit partir des données originales.
- Séparez immédiatement en train (70 %) et test (30 %) en précisant un random state pour la reproducibilité. Isolez `SalePrice` dans `y_train` et `y_test`.

> `train_test_split` de `sklearn.model_selection` fait ce split. Après le split, toute transformation doit être apprise uniquement sur `X_train`. Ne revenez jamais en arrière pour recalculer quelque chose sur l'ensemble complet.
- Identifiez les types de variables dans `X_train` :
   - Variables **numériques** (int, float)
   - Variables **catégorielles** (object): mais aussi certaines colonnes `int` comme `MSSubClass` qui encodent des catégories



**Dans une cellule Markdown, répondez :**
- Combien de variables numériques et catégorielles compte le dataset ?
- `MSSubClass` est codé comme un entier: pourquoi serait-il incorrect de le traiter comme une variable numérique dans un modèle ?
- Citez deux observations de votre EDA (exercices précédents) qui vont directement guider vos choix de transformation dans cet exercice.



---
## Section 2: Encodage des variables catégorielles

**Objectif** : transformer les variables qualitatives en représentations numériques adaptées à leur nature.

> Rappel des trois situations à distinguer :
> - Variable **nominale** (sans ordre) : One-Hot Encoding
> - Variable **ordinale** (avec ordre naturel) : Ordinal Encoding avec ordre défini manuellement
> - Variable nominale à **haute cardinalité** (>10 modalités) : Target Encoding ou Frequency Encoding



### 2.1 One-Hot Encoding sur `Neighborhood`

**Ce que vous devez faire :**
- Appliquez un One-Hot Encoding sur `Neighborhood` (environ 25 quartiers).

> `OneHotEncoder` de `sklearn.preprocessing` s'utilise avec `fit()` sur `X_train` puis `transform()` sur `X_test`. Le paramètre `drop='first'` supprime une colonne pour éviter la dummy variable trap (multicolinéarité parfaite). Le paramètre `sparse_output=False` retourne un tableau dense plus facile à manipuler.
- Affichez le nombre de colonnes obtenues avant et après encodage. Que se passe-t-il si une modalité de `Neighborhood` n'apparaît pas dans le train set mais apparaît dans le test set ?

> `handle_unknown='ignore'` dans `OneHotEncoder` gère ce cas: les modalités inconnues produisent une ligne de zéros au lieu d'une erreur.


**Dans une cellule Markdown, répondez :**
- Pourquoi one-hot encoder une variable nominale plutôt que de simplement lui attribuer des entiers (0, 1, 2...) ?
- Pour un dataset avec 100 quartiers, le one-hot encoding créerait 99 colonnes. Quelles alternatives envisageriez-vous ?


### 2.2 Ordinal Encoding sur `ExterQual`

**Ce que vous devez faire :**
- `ExterQual` (qualité extérieure) prend les valeurs `Po`, `Fa`, `TA`, `Gd`, `Ex`: dans cet ordre croissant de qualité. Appliquez un encodage ordinal en respectant cet ordre.

> `OrdinalEncoder` de `sklearn.preprocessing` accepte un paramètre `categories` où vous passez explicitement l'ordre. Si vous ne spécifiez pas l'ordre, l'encodeur utilise l'ordre alphabétique: ce qui serait incorrect ici.
- Affichez quelques lignes avant et après pour vérifier que l'ordre est bien respecté.




**Dans une cellule Markdown, répondez :**
- Pourquoi est-il important de définir l'ordre manuellement plutôt que de laisser l'encodeur décider ?
- Donnez un exemple de variable du dataset qui serait incorrectement encodée si on utilisait l'ordre alphabétique.


### 2.3 Target Encoding sur `MSSubClass`

**Ce que vous devez faire :**
- `MSSubClass` représente le type de construction (16 modalités codées en entiers mais sans signification ordinale). Appliquez un Target Encoding.

> `TargetEncoder` est disponible dans `sklearn.preprocessing` depuis scikit-learn 1.3. Il doit être entraîné avec la cible (`y_train`) en plus de `X_train`. Appelez `fit(X_train[['MSSubClass']], y_train)` puis `transform(X_test[['MSSubClass']])`. Le mécanisme de cross-fitting intégré protège contre la fuite de données.

- Affichez la valeur encodée pour quelques modalités. Que remarquez-vous pour une modalité très rare (peu d'observations) ?



**Dans une cellule Markdown, répondez :**
- Pourquoi le target encoding est-il particulièrement risqué pour les petites catégories ?
- Quelle différence entre target encoding et frequency encoding ? Dans quelle situation préféreriez-vous l'un à l'autre ?



### 2.4 Frequency Encoding sur une variable de votre choix

**Ce que vous devez faire :**
- Choisissez une variable catégorielle nominale avec une cardinalité modérée (entre 5 et 15 modalités) et appliquez un frequency encoding manuellement.

> Le frequency encoding se calcule simplement : `freq_map = X_train['col'].value_counts()` donne les fréquences sur le train. Utilisez ensuite `.map(freq_map)` pour encoder train ET test. Attention : les modalités du test absentes du train recevront `NaN`: gérez ce cas.





**Dans une cellule Markdown, répondez :**
- Quel problème surgit si deux catégories ont exactement la même fréquence ?
- Ce problème est-il grave pour un modèle de régression ? Pour un modèle de classification ?


---
## Section 3: Transformation des distributions asymétriques

**Objectif** : réduire le skewness des variables numériques pour améliorer la performance des modèles linéaires.

> Rappel de l'EDA : vous avez calculé le skewness de `SalePrice` (≈ 1.78) et `LotArea` (≈ 12.8) dans l'Exercice 1.1. Ces deux variables sont de bons candidats à la transformation.



### 3.1 Analyse préliminaire

**Ce que vous devez faire :**
- Calculez le skewness de toutes les variables numériques de `X_train` et identifiez celles avec |skewness| > 1.

> `.skew()` calcule le skewness par colonne. Filtrez les résultats pour ne garder que les colonnes fortement asymétriques.
- Pour `SalePrice` (la cible) et `LotArea`, tracez un histogramme et un Q-Q plot côte à côte.

> `scipy.stats.probplot()` génère le Q-Q plot. Pour une distribution normale, les points doivent suivre la diagonale. Un écart marqué confirme l'asymétrie.

### 3.2 Log-transform

**Ce que vous devez faire :**
-  Appliquez `np.log1p()` sur `SalePrice` (cible) et `LotArea` (feature). Tracez les distributions avant et après, et recalculez le skewness.

> `np.log1p(x)` calcule log(1 + x): plus sûr que `np.log(x)` car il ne produit pas d'erreur pour les valeurs égales à 0. Comparez `df['LotArea'].skew()` avant et après transformation.

> `SalePrice` est la cible: transformez `y_train` et `y_test` séparément, pas depuis le DataFrame. Quand vous évaluerez le modèle, pensez à **inverser la transformation** sur les prédictions avant de calculer la RMSE, sinon vous comparez des log-prix et des prix en dollars.

### 3.3 Yeo-Johnson

**Ce que vous devez faire :**
-  Appliquez une transformation Yeo-Johnson sur `LotArea` et comparez le résultat au log-transform.

> `PowerTransformer(method='yeo-johnson')` de `sklearn.preprocessing` s'utilise comme un scaler standard (`fit` sur train, `transform` sur test). L'avantage sur Box-Cox : il fonctionne aussi avec des valeurs négatives ou nulles. Comparez le skewness obtenu avec les deux méthodes.


**Dans une cellule Markdown, répondez :**
- Laquelle des deux transformations (log ou Yeo-Johnson) réduit davantage le skewness de `LotArea` ?
- Pourquoi est-il important de `fit` le `PowerTransformer` sur le train uniquement ?
- Quelles variables de votre liste (skewness > 1) allez-vous transformer dans le pipeline final ? Justifiez vos choix.


---
## Section 4: Traitement des valeurs manquantes

**Objectif** : gérer les valeurs manquantes sans fuite de données et sans perdre d'information utile.

> Rappel de l'EDA : certaines valeurs manquantes dans Ames Housing ne signifient pas "information inconnue" mais "cette caractéristique n'existe pas" (ex. `PoolQC` manquant = pas de piscine). Ces variables doivent être traitées différemment des vraies valeurs manquantes.



### 4.1 Inventaire et décisions

**Ce que vous devez faire :**

- Calculez le pourcentage de valeurs manquantes par colonne dans `X_train`. Identifiez trois catégories :
   - Colonnes avec > 80 % de manquants → suppression
   - Colonnes avec manquants "structurels" (absence de caractéristique) → créer une catégorie "Aucun" ou valeur 0
   - Colonnes avec manquants aléatoires → imputation

**Dans une cellule Markdown, construisez ce tableau de décisions :**

| Variable | % manquants | Signification du manquant | Stratégie choisie | Justification |
|----------|-------------|--------------------------|-------------------|---------------|
| `PoolQC` | ... | Pas de piscine | Catégorie "Aucun" | Valeur manquante structurelle |
| ... | ... | ... | ... | ... |



### 4.2 Imputation simple

**Ce que vous devez faire :**
- Pour les variables numériques avec manquants aléatoires, appliquez une imputation par la **médiane**.
- Pour les variables catégorielles avec manquants aléatoires, appliquez une imputation par la **valeur la plus fréquente**.

> `SimpleImputer` de `sklearn.impute` accepte un paramètre `strategy` : `'median'`, `'mean'`, `'most_frequent'` ou `'constant'`. Appliquez `fit()` sur `X_train` et `transform()` sur `X_test`.
- Sur une variable imputée, affichez les statistiques descriptives avant et après imputation. La médiane a-t-elle changé ? L'écart-type ?



### 4.3 KNN Imputer

**Ce que vous devez faire :**
- Appliquez un `KNNImputer` sur un sous-ensemble de 5 variables numériques qui ont des valeurs manquantes.

> `KNNImputer` de `sklearn.impute` cherche les k voisins les plus proches (défaut k=5) et impute avec leur moyenne. Comme il est basé sur des distances, **les variables doivent être mises à l'échelle d'abord**. Créez un mini-pipeline `StandardScaler` + `KNNImputer` pour ce sous-ensemble.
- Comparez les valeurs imputées par KNN et par SimpleImputer (médiane) pour quelques observations. Notez les différences dans une cellule Markdown.



### 4.4 Indicateurs de manquants

**Ce que vous devez faire :**
- Pour les variables où le fait d'être manquant est potentiellement informatif (ex. `BsmtQual`: une maison sans sous-sol vs une maison avec un sous-sol dont la qualité n'est pas renseignée), créez une colonne binaire indicatrice.

> `MissingIndicator` de `sklearn.impute` crée automatiquement ces colonnes binaires. Vous pouvez aussi le faire manuellement avec `df['col_missing'] = df['col'].isna().astype(int)`: mais l'avantage de `MissingIndicator` est qu'il s'intègre proprement dans un pipeline.



**Dans une cellule Markdown, répondez :**
- Pourquoi ne pas imputer après le split train/test serait-il une fuite de données ?
- Pour `GarageYrBlt` (année de construction du garage), quelle stratégie d'imputation vous semble la plus adaptée et pourquoi ?


## Section 5: Normalisation et mise à l'échelle

**Objectif** : comparer l'effet de trois scalers sur des variables contenant des outliers.

> Rappel du cours : les arbres de décision et les forêts aléatoires sont insensibles à l'échelle. Mais Lasso (régression linéaire régularisée) est sensible: les variables avec de grandes valeurs domineront sinon.

**Ce que vous devez faire :**
- Sélectionnez 5 variables numériques déjà imputées : `GrLivArea`, `LotArea`, `TotalBsmtSF`, `1stFlrSF`, `YearBuilt`.
- Appliquez successivement les trois scalers sur ces variables et visualisez les résultats :
    - `StandardScaler`
    - `MinMaxScaler`
    - `RobustScaler`

> Pour chaque scaler, `fit()` sur `X_train`, `transform()` sur `X_train` et `X_test`. Pour comparer visuellement, tracez des boxplots côte à côte des 5 variables après chaque scaler (3 figures au total, ou une figure avec 3 lignes de boxplots).
- Sur `LotArea` (qui contient des outliers extrêmes), affichez les valeurs minimale et maximale après chaque scaler. Quelle différence observez-vous ?




**Dans une cellule Markdown, répondez :**
- Pourquoi `MinMaxScaler` est-il particulièrement sensible aux outliers de `LotArea` ?
- `RobustScaler` utilise la médiane et l'IQR: en quoi cela le rend-il plus robuste ?
- Pour le pipeline final avec Lasso, quel scaler allez-vous choisir et pourquoi ? Prenez en compte la présence d'outliers dans Ames Housing.


---
## Section 6: Pipeline complet et évaluation

**Objectif** : mesurer quantitativement l'impact de vos choix de feature engineering sur la performance d'un modèle Lasso.

> `Pipeline` et `ColumnTransformer` de scikit-learn permettent d'enchaîner toutes vos transformations sans risque de fuite de données. Le `ColumnTransformer` applique des transformations différentes selon le type de colonne (numériques / catégorielles). Le `Pipeline` enchaîne preprocessing + modèle.



### 6.1 Pipeline baseline (sans feature engineering)

**Ce que vous devez faire :**

- Construisez un pipeline minimal :
    - Imputation par la médiane pour les numériques, par la fréquence pour les catégorielles
    - One-hot encoding basique pour toutes les catégorielles
    - StandardScaler pour les numériques
    - Modèle Lasso avec `alpha=1.0`

- Entraînez sur `X_train` / `y_train`, prédisez sur `X_test`, calculez la **RMSE**.

> `mean_squared_error(y_test, y_pred, squared=False)` calcule la RMSE. Si vous avez log-transformé `y_train`, n'oubliez pas d'appliquer `np.expm1()` (l'inverse de `log1p`) sur les prédictions avant de calculer la RMSE en dollars.



### 6.2 Pipeline avec feature engineering complet

**Ce que vous devez faire :**
- Construisez un second pipeline intégrant toutes vos décisions de l'exercice :
    - Colonnes à supprimer (trop de manquants)
    - Imputation différenciée selon la nature du manquant
    - Indicateurs de manquants pour les variables appropriées
    - Ordinal encoding pour les variables ordinales (avec ordre défini)
    - Target encoding pour les variables à haute cardinalité
    - One-hot encoding pour les variables nominales à cardinalité modérée
    - Log-transform ou Yeo-Johnson pour les variables asymétriques
    - RobustScaler pour les variables numériques

- Entraînez et calculez la RMSE.


### 6.3 Comparaison et analyse

**Ce que vous devez produire :**
-  Un tableau comparatif :

| Pipeline | RMSE (en $) | Observations |
|----------|-------------|--------------|
| Baseline (sans FE) | | |
| Feature Engineering complet | | |


**Dans une cellule Markdown, répondez :**
- Quelle transformation a probablement eu le plus d'impact ? Comment pourriez-vous le vérifier isolément ?
- Si le pipeline FE est moins performant que le baseline sur certains aspects, proposez une explication.
- Le Lasso avec `alpha=1.0` est arbitraire. Comment optimiseriez-vous ce paramètre sans toucher au test set ?


## Checklist avant de rendre

- [ ] Le notebook s'exécute du début à la fin sans erreur (`Restart & Run All`)
- [ ] Le split train/test est effectué en Section 1 et aucune transformation n'est `fit` sur le test set
- [ ] La Section 2 contient les 4 types d'encodage avec justification du choix pour chaque variable
- [ ] Le tableau de décisions des valeurs manquantes (Section 4.1) est complété
- [ ] Les 3 scalers sont comparés visuellement sur les mêmes variables
- [ ] Les deux pipelines (baseline et FE complet) produisent chacun une RMSE valide
- [ ] Si `y_train` a été log-transformé, `np.expm1()` est appliqué avant le calcul de RMSE
- [ ] Chaque section se termine par des réponses aux questions Markdown

---

